In [59]:
print(9*9)

81


In [60]:

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.embeddings import FakeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

In [61]:
import os
# GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
INDEX_NAME = "banking-policy-index" # Ensure this index exists in your Pinecone console


In [62]:
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/embedding-001", 
#     google_api_key=GOOGLE_API_KEY
# )

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 

In [63]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists, if not, create it
if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    print(f"Index '{INDEX_NAME}' not found. Creating it now...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=384, # MUST match the embedding model (384 for MiniLM, 768 for Google)
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1" # Starter plan usually requires us-east-1
        )
    )
    # Wait until the index is fully initialized and ready to receive data
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print("Waiting for Pinecone index to initialize...")
        time.sleep(2)
    print("Index created successfully!")

# Now initialize the VectorStore safely

# embeddings = FakeEmbeddings(size=768)
# vectorstore = InMemoryVectorStore(embeddings)
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=PINECONE_API_KEY
)

In [64]:
pdf_content = [
    "Banking Policy 101: If Child 1 picks a prime number, the Father must be extra formal.",
    "Banking Policy 102: Odd numbers are considered 'High Volatility' in this simulation.",
    "Banking Policy 103: Even numbers are considered 'Stable Assets'."
]

vectorstore.add_texts(pdf_content)

['88463768-da41-45a4-a918-313b30075ff1',
 '5c6cd4b6-c126-4a93-ae2e-791c1dc6bae7',
 '90f671dd-98e4-41d7-8b81-01a28a0e422b']

In [71]:
# 2. DEFINE THE TOOL
@tool
def lookup_policy(query: str, k:int = 2):
    """Consult the internal banking policy PDF to get context on numbers."""
    docs = vectorstore.similarity_search(query, k)
    return docs[0].page_content

tools = [lookup_policy]

llm_with_tools = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
).bind_tools(tools)

In [72]:
query = "According to the bank policy what are considered 'Stable assets'? And what are numbers are called Highly volatile?"
messages = [HumanMessage(content=query )]
ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response)
print(ai_response.content)

In [73]:
tool_response = ai_response.tool_calls
tool_response

[{'name': 'lookup_policy',
  'args': {'query': "What are considered 'Stable assets'"},
  'id': '992f6e71-3a87-4204-8258-8e73741574eb',
  'type': 'tool_call'},
 {'name': 'lookup_policy',
  'args': {'query': 'What numbers are considered Highly volatile'},
  'id': 'dd6c6675-ae60-424f-87b2-c05ab6fafe06',
  'type': 'tool_call'}]

In [74]:
#loop thru all tool calls
for tool_call in tool_response:

    if tool_call:
        tool_args = tool_call["args"]
        tool_query = tool_call["args"]["query"]
        tool_id = tool_call["id"]

        tool_output = lookup_policy.invoke(tool_args)
        
        tool_msg = ToolMessage(content=tool_output, tool_call_id=tool_id)
        messages.append(tool_msg)


In [75]:
# final tool-info powered llm call
response = llm_with_tools.invoke(messages)
print(response.content)

In [76]:
tool_msg

ToolMessage(content="Banking Policy 102: Odd numbers are considered 'High Volatility' in this simulation.", tool_call_id='dd6c6675-ae60-424f-87b2-c05ab6fafe06')

In [78]:
messages

[HumanMessage(content="According to the bank policy what are considered 'Stable assets'? And what are numbers are called Highly volatile?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'lookup_policy', 'arguments': '{"query": "What numbers are considered Highly volatile"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bf72e-1f35-7f52-99ae-622a8d0dcf72-0', tool_calls=[{'name': 'lookup_policy', 'args': {'query': "What are considered 'Stable assets'"}, 'id': '992f6e71-3a87-4204-8258-8e73741574eb', 'type': 'tool_call'}, {'name': 'lookup_policy', 'args': {'query': 'What numbers are considered Highly volatile'}, 'id': 'dd6c6675-ae60-424f-87b2-c05ab6fafe06', 'type': 'tool_call'}], usage_metadata={'input_tokens': 81, 'output_tokens': 41, 'total_tokens': 122, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(co